In [17]:
from pathlib import Path
import importlib.util
import json
import os
import random
import shutil
import subprocess
import sys
from collections import Counter
from dataclasses import dataclass

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CACHE_ROOT = PROJECT_ROOT / ".cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("YOLO_CONFIG_DIR", str(CACHE_ROOT / "ultralytics"))
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_ROOT / "matplotlib"))
os.environ.setdefault("KAGGLEHUB_CACHE", str(CACHE_ROOT / "kagglehub"))
for folder in ["ultralytics", "matplotlib", "kagglehub"]:
    (CACHE_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("Python version:", sys.version)
print("Project root:", PROJECT_ROOT)

if sys.version_info[:2] != (3, 14):
    print("WARNING: Notebook dang khong chay Python 3.14. Hay chon kernel Python 3.14 trong VS Code neu ban muon dung 3.14.")

# Windows Application Control / Smart App Control can block torch DLL/PYD files in Python 3.14 user site-packages.
# Try to remove Mark-of-the-Web streams before importing torch/ultralytics.
def unblock_torch_windows_files():
    if os.name != "nt":
        return
    candidates = []
    for site_root in [Path(p) for p in sys.path if "site-packages" in p.lower()]:
        torch_root = site_root / "torch"
        if torch_root.exists():
            candidates.append(torch_root)
    for torch_root in candidates:
        files = list(torch_root.rglob("*.dll")) + list(torch_root.rglob("*.pyd"))
        if not files:
            continue
        try:
            subprocess.run(
                ["powershell", "-NoProfile", "-ExecutionPolicy", "Bypass", "-Command", f"Get-ChildItem -LiteralPath '{torch_root}' -Recurse -Include *.dll,*.pyd | Unblock-File"],
                check=False,
                capture_output=True,
                text=True,
            )
            print(f"Checked/unblocked torch binary files in: {torch_root}")
        except Exception as exc:
            print("Could not run Unblock-File for torch binaries:", exc)

unblock_torch_windows_files()

REQUIRED_PACKAGES = {
    "cv2": "opencv-python",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "yaml": "PyYAML",
    "ultralytics": "ultralytics",
    "kagglehub": "kagglehub",
}

missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print("Installing missing packages into current Python 3.14 kernel:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    unblock_torch_windows_files()

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import HTML, display

try:
    from ultralytics import YOLO
except OSError as exc:
    message = str(exc)
    if "Application Control policy" in message or "torch_global_deps.dll" in message:
        raise RuntimeError(
            "Python 3.14 dang bi Windows Application Control chan torch DLL. "
            "Notebook da thu Unblock-File nhung chua thanh cong. "
            "Hay chay PowerShell: Get-ChildItem $env:LOCALAPPDATA\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\torch -Recurse -Include *.dll,*.pyd | Unblock-File "
            "roi restart kernel, hoac cai lai torch trong moi truong Python 3.14 duoc Windows tin cay."
        ) from exc
    raise

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)


Python: /usr/bin/python3
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Project root: /content


## 1.1. Utility Helpers

Nhung helper nay thay the script `.py` ben ngoai de notebook tu chua toan bo logic xu ly.

In [18]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def display_styled_table(df: pd.DataFrame, caption: str | None = None, max_rows: int = 20):
    view = df.head(max_rows).copy()
    styler = view.style.set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "16px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("background", "#1f2937"), ("color", "white")]},
    ])
    if caption:
        styler = styler.set_caption(caption)
    display(styler)

def write_yaml(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")

def read_yaml(path: Path) -> dict:
    return yaml.safe_load(path.read_text(encoding="utf-8")) or {}

def normalize_names(names) -> dict[int, str]:
    if isinstance(names, list):
        return {i: str(name) for i, name in enumerate(names)}
    return {int(k): str(v) for k, v in names.items()}

def link_or_copy(source: Path, destination: Path, use_copy: bool = False):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        destination.unlink()
    if use_copy:
        shutil.copy2(source, destination)
        return
    try:
        destination.hardlink_to(source)
    except OSError:
        shutil.copy2(source, destination)

def label_path_for_image(image_path: Path, images_dir: Path) -> Path:
    relative = image_path.relative_to(images_dir)
    return images_dir.parent.parent / "labels" / images_dir.name / relative.with_suffix(".txt")

def read_yolo_label(label_path: Path) -> list[dict]:
    rows = []
    if not label_path.exists():
        return rows
    for line_no, raw_line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), 1):
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            rows.append({"error": f"invalid format at line {line_no}: {line}"})
            continue
        try:
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])
            rows.append({
                "class_id": class_id,
                "x_center": x_center,
                "y_center": y_center,
                "width": width,
                "height": height,
                "area": width * height,
            })
        except ValueError:
            rows.append({"error": f"non numeric label at line {line_no}: {line}"})
    return rows

def resolve_split_dir(data_yaml: Path, split: str) -> Path:
    cfg = read_yaml(data_yaml)
    root = Path(cfg.get("path", data_yaml.parent))
    if not root.is_absolute():
        root = (data_yaml.parent / root).resolve()
    split_path = Path(cfg[split])
    return split_path if split_path.is_absolute() else root / split_path

## 1.2. End-to-End Processing Flow

In [19]:
display(HTML("""
<div style="font-family:Arial; max-width:900px;">
  <div style="padding:12px; background:#e8f1ff; border:1px solid #9db7df; border-radius:10px; text-align:center;color:#000000;"><b>Data Collection</b></div>
  <div style="text-align:center; font-size:24px;">&#8595;</div>
  <div style="padding:12px; background:#e9f8ef; border:1px solid #96c5a4; border-radius:10px; text-align:center;color:#000000;"><b>Exploring Data Analysis</b></div>
  <div style="text-align:center; font-size:24px;">&#8595;</div>
  <div style="padding:12px; background:#f2efff; border:1px solid #aea6db; border-radius:10px; text-align:center;color:#000000;"><b>Data Cleaning</b></div>
  <div style="text-align:center; font-size:24px;">&#8595;</div>
  <div style="padding:12px; background:#fff5df; border:1px solid #d6bc7b; border-radius:10px; text-align:center;color:#000000;"><b>Data Preparing</b></div>
  <div style="text-align:center; font-size:24px;">&#8595;</div>
  <div style="padding:12px; background:#fdecea; border:1px solid #d9aaa2; border-radius:10px; text-align:center;color:#000000;"><b>Model Training</b></div>
  <div style="text-align:center; font-size:24px;">&#8595;</div>
  <div style="padding:12px; background:#edf7e7; border:1px solid #a8c991; border-radius:10px; text-align:center;color:#000000;"><b>Evaluation</b></div>
  <div style="text-align:center; font-size:24px;">&#8595;</div>
  <div style="padding:12px; background:#fce7f3; border:1px solid #d59ab8; border-radius:10px; text-align:center;color:#000000;"><b>Inference</b></div>
</div>
"""))

## **`2. Data Collection`**

Tai dataset Kaggle va chuan hoa sang YOLO structure.

In [20]:
from pathlib import Path
import kagglehub, random, shutil

KAGGLE_DATASET = "ahemateja19bec1025/traffic-sign-dataset-classification"

DATA_ROOT = Path("/content/data")
CLS_ROOT = DATA_ROOT / "processed" / "traffic-sign-classification"

downloaded_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("Downloaded path:", downloaded_path)

def has_images(folder):
    return any(p.suffix.lower() in IMAGE_EXTENSIONS for p in folder.iterdir() if p.is_file())

def find_class_root(root):
    candidates = []
    for p in [root, *[x for x in root.rglob("*") if x.is_dir()]]:
        class_dirs = [d for d in p.iterdir() if d.is_dir() and has_images(d)]
        if len(class_dirs) >= 2:
            image_count = sum(len([f for f in d.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]) for d in class_dirs)
            candidates.append((p, image_count))
    if not candidates:
        raise FileNotFoundError("Không tìm thấy folder dạng class_name/*.jpg")
    return max(candidates, key=lambda x: x[1])[0]

CLASS_SOURCE_ROOT = find_class_root(downloaded_path)
print("Class source root:", CLASS_SOURCE_ROOT)

if CLS_ROOT.exists():
    shutil.rmtree(CLS_ROOT)

rng = random.Random(42)
rows = []

for class_dir in sorted([d for d in CLASS_SOURCE_ROOT.iterdir() if d.is_dir()]):
    images = [p for p in class_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
    if not images:
        continue

    rng.shuffle(images)
    n = len(images)
    n_val = max(1, int(n * 0.15))
    n_test = max(1, int(n * 0.15))

    split_items = {
        "test": images[:n_test],
        "val": images[n_test:n_test + n_val],
        "train": images[n_test + n_val:],
    }

    for split, paths in split_items.items():
        for i, src in enumerate(paths):
            dst = CLS_ROOT / split / class_dir.name / f"{i:05d}_{src.name}"
            link_or_copy(src, dst)
            rows.append({"split": split, "class_name": class_dir.name, "image_path": dst})

images_df = pd.DataFrame(rows)
display_styled_table(images_df.groupby(["split", "class_name"]).size().reset_index(name="images"), "Classification Dataset")

Using Colab cache for faster access to the 'traffic-sign-dataset-classification' dataset.
Downloaded path: /kaggle/input/traffic-sign-dataset-classification
Class source root: /kaggle/input/traffic-sign-dataset-classification/traffic_Data/DATA


,split,class_name,images
0,test,0,17
1,test,1,6
2,test,10,10
3,test,11,20
4,test,12,14
5,test,13,5
6,test,14,19
7,test,15,3
8,test,16,21
9,test,17,19


In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **2.1. Convert Kaggle Dataset to YOLO Split**

In [22]:
def has_images(folder: Path) -> bool:
    return any(
        p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        for p in folder.iterdir()
    )

def find_classification_root(root: Path) -> Path:
    root = Path(root)
    candidates = []

    for path in [root, *[p for p in root.rglob("*") if p.is_dir()]]:
        class_dirs = [
            d for d in path.iterdir()
            if d.is_dir() and has_images(d)
        ]
        if len(class_dirs) >= 2:
            image_count = sum(
                len([f for f in d.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS])
                for d in class_dirs
            )
            candidates.append((path, image_count))

    if not candidates:
        raise FileNotFoundError("Khong tim thay cau truc classification dang class_name/*.jpg")

    return max(candidates, key=lambda item: item[1])[0]


CLASS_SOURCE_ROOT = find_classification_root(downloaded_path)
CLASSIFICATION_ROOT = DATA_ROOT / "processed" / "traffic-sign-classification"

if CLASSIFICATION_ROOT.exists():
    shutil.rmtree(CLASSIFICATION_ROOT)

rng = random.Random(42)
records = []

for class_dir in sorted([p for p in CLASS_SOURCE_ROOT.iterdir() if p.is_dir()]):
    images = sorted([
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])

    if not images:
        continue

    rng.shuffle(images)

    n = len(images)
    val_count = max(1, int(n * 0.15))
    test_count = max(1, int(n * 0.15))

    split_map = {
        "test": images[:test_count],
        "val": images[test_count:test_count + val_count],
        "train": images[test_count + val_count:],
    }

    for split, split_images in split_map.items():
        for index, source in enumerate(split_images):
            target = CLASSIFICATION_ROOT / split / class_dir.name / f"{index:05d}_{source.name}"
            link_or_copy(source, target)
            records.append({
                "split": split,
                "class_name": class_dir.name,
                "image_path": str(target),
            })

images_df = pd.DataFrame(records)

display_styled_table(
    images_df.groupby(["split", "class_name"]).size().reset_index(name="images"),
    "Classification Dataset",
    120,
)

RUNTIME_DATA_ROOT = CLASSIFICATION_ROOT
print("Classification dataset root:", RUNTIME_DATA_ROOT)

NameError: name 'KAGGLE_ARCHIVE' is not defined

## **`3. Exploratory Data Analysis (EDA)`**

In [ ]:
cfg = read_yaml(DATA_YAML)
names = normalize_names(cfg["names"])
dataset_root = Path(cfg["path"])
if not dataset_root.is_absolute():
    dataset_root = (DATA_YAML.parent / dataset_root).resolve()

display_styled_table(pd.DataFrame({"class_id": list(names.keys()), "class_name": list(names.values())}), "Class Map", 60)
print("Dataset root:", dataset_root)
print("Classes:", len(names))

In [ ]:
RUNTIME_DATA_ROOT = CLS_ROOT

images_df = pd.DataFrame([
    {"split": split, "class_name": class_dir.name, "image_path": image_path}
    for split in ["train", "val", "test"]
    for class_dir in sorted((RUNTIME_DATA_ROOT / split).iterdir())
    if class_dir.is_dir()
    for image_path in class_dir.iterdir()
    if image_path.suffix.lower() in IMAGE_EXTENSIONS
])

display_styled_table(
    images_df.groupby(["split", "class_name"]).size().reset_index(name="images"),
    "Class Distribution",
    120,
)

In [ ]:
valid_labels_df = labels_df[labels_df.get("error").isna()] if "error" in labels_df.columns else labels_df.copy()
class_counts = valid_labels_df.groupby(["class_id", "class_name"]).size().reset_index(name="objects").sort_values("objects", ascending=False)
display_styled_table(class_counts, "Class Distribution", 60)

plt.figure(figsize=(10, max(8, len(class_counts) * 0.25)))
plot_df = class_counts.sort_values("objects")
plt.barh(plot_df["class_name"], plot_df["objects"])
plt.title("Class distribution")
plt.xlabel("Objects")
plt.tight_layout()
plt.show()

## **`4. DATA CLEANING`**

L

In [ ]:
cfg = read_yaml(DATA_YAML)
names = normalize_names(cfg["names"])

BASIC_CLEAN_ROOT = DATA_ROOT / "processed" / "vietnamese-traffic-signs-kaggle-basic-clean"
CLEAN_ROOT = DATA_ROOT / "processed" / "vietnamese-traffic-signs-kaggle-clean"
CLEAN_YAML = DATA_ROOT / "data.clean.yaml"
MIN_BBOX_AREA = 0.00005

def validate_label_file(label_path: Path, class_count: int):
    valid_lines, reasons = [], []
    if not label_path.exists():
        return valid_lines, ["missing_label"]
    for line_no, raw_line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), 1):
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            reasons.append(f"invalid_format:{line_no}")
            continue
        try:
            class_id = int(parts[0])
            x, y, w, h = map(float, parts[1:])
        except ValueError:
            reasons.append(f"non_numeric:{line_no}")
            continue
        if class_id < 0 or class_id >= len(names):
            reasons.append(f"class_out_of_range:{line_no}")
            continue
        if any(v < 0 or v > 1 for v in [x, y, w, h]):
            reasons.append(f"bbox_outside_0_1:{line_no}")
            continue
        if w <= 0 or h <= 0 or w * h < MIN_BBOX_AREA:
            reasons.append(f"bbox_invalid_or_too_small:{line_no}")
            continue
        valid_lines.append(f"{class_id} {x:.8f} {y:.8f} {w:.8f} {h:.8f}")
    if not valid_lines:
        reasons.append("empty_or_no_valid_labels")
    return valid_lines, reasons

def sha256_file(path: Path):
    import hashlib
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def reset_yolo_dirs(root: Path):
    for split in ["train", "val", "test"]:
        for folder in [root / "images" / split, root / "labels" / split]:
            if folder.exists():
                shutil.rmtree(folder)
            folder.mkdir(parents=True, exist_ok=True)

reset_yolo_dirs(BASIC_CLEAN_ROOT)

seen_hashes = set()
clean_report = {"summary": {}, "rejected": []}
for split in ["train", "val", "test"]:
    images_dir = resolve_split_dir(DATA_YAML, split)
    image_paths = sorted(path for path in images_dir.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)
    kept = 0
    reasons_counter = Counter()
    for image_path in image_paths:
        label_path = label_path_for_image(image_path, images_dir)
        reasons = []
        image = cv2.imread(str(image_path))
        if image is None:
            reasons.append("unreadable_image")
        valid_lines, label_reasons = validate_label_file(label_path, len(names))
        reasons.extend(label_reasons)
        image_hash = None
        if image is not None:
            image_hash = sha256_file(image_path)
            if image_hash in seen_hashes:
                reasons.append("duplicate_file")
        if reasons:
            reasons_counter.update(reasons)
            clean_report["rejected"].append({"split": split, "image": str(image_path), "reason": ";".join(reasons)})
            continue
        link_or_copy(image_path, BASIC_CLEAN_ROOT / "images" / split / image_path.name)
        out_label = BASIC_CLEAN_ROOT / "labels" / split / label_path.name
        out_label.write_text("\n".join(valid_lines) + "\n", encoding="utf-8")
        seen_hashes.add(image_hash)
        kept += 1
    clean_report["summary"][split] = {"total": len(image_paths), "kept": kept, "removed": len(image_paths) - kept, "reasons": dict(reasons_counter)}

basic_cfg = dict(cfg)
basic_cfg["path"] = str(BASIC_CLEAN_ROOT).replace("\\", "/")
basic_cfg["train"] = "images/train"
basic_cfg["val"] = "images/val"
basic_cfg["test"] = "images/test"
BASIC_CLEAN_YAML = DATA_ROOT / "data.basic-clean.yaml"
write_yaml(BASIC_CLEAN_YAML, basic_cfg)

# Keep only classes that actually have training data, then remap ids to 0..N-1.
basic_images_df, basic_labels_df = scan_yolo_dataset(BASIC_CLEAN_YAML)
valid_basic_labels = basic_labels_df[basic_labels_df.get("error").isna()] if "error" in basic_labels_df.columns else basic_labels_df
train_active_ids = sorted(valid_basic_labels.loc[valid_basic_labels["split"] == "train", "class_id"].dropna().astype(int).unique().tolist())
if not train_active_ids:
    raise ValueError("No active train classes found after cleaning.")

old_to_new = {old_id: new_id for new_id, old_id in enumerate(train_active_ids)}
new_names = {new_id: names[old_id] for old_id, new_id in old_to_new.items()}
removed_classes = [{"old_class_id": old_id, "class_name": names[old_id]} for old_id in names if old_id not in old_to_new]

reset_yolo_dirs(CLEAN_ROOT)
remap_report = {"kept_images": {}, "removed_images": {}, "removed_classes": removed_classes}
for split in ["train", "val", "test"]:
    kept = 0
    removed = 0
    images_dir = BASIC_CLEAN_ROOT / "images" / split
    for image_path in sorted(path for path in images_dir.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS):
        label_path = label_path_for_image(image_path, images_dir)
        new_lines = []
        for label in read_yolo_label(label_path):
            if "error" in label:
                continue
            old_id = int(label["class_id"])
            if old_id not in old_to_new:
                continue
            new_lines.append(f"{old_to_new[old_id]} {label['x_center']:.8f} {label['y_center']:.8f} {label['width']:.8f} {label['height']:.8f}")
        if not new_lines:
            removed += 1
            continue
        link_or_copy(image_path, CLEAN_ROOT / "images" / split / image_path.name)
        (CLEAN_ROOT / "labels" / split / label_path.name).write_text("\n".join(new_lines) + "\n", encoding="utf-8")
        kept += 1
    remap_report["kept_images"][split] = kept
    remap_report["removed_images"][split] = removed

clean_cfg = {
    "path": str(CLEAN_ROOT).replace("\\", "/"),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": new_names,
}
write_yaml(CLEAN_YAML, clean_cfg)
(CLEAN_ROOT / "filter_report.json").write_text(json.dumps(clean_report, indent=2, ensure_ascii=False), encoding="utf-8")
(CLEAN_ROOT / "remap_report.json").write_text(json.dumps(remap_report, indent=2, ensure_ascii=False), encoding="utf-8")

cfg = read_yaml(CLEAN_YAML)
names = normalize_names(cfg["names"])
display_styled_table(pd.DataFrame([{"split": k, **v} for k, v in clean_report["summary"].items()]), "Cleaning Summary")
display_styled_table(pd.DataFrame(remap_report["removed_classes"]), "Removed Classes Without Train Data", 80)
print("Active train classes after remap:", len(names))
print("Clean dataset:", CLEAN_YAML)

## **`5. DATA PREPARING`**

Section nay tao dataset train that cho YOLO. Train split duoc can bang hon bang cach cap class qua nhieu va oversample class qua it. Val/test giu theo clean data sau remap de evaluation trung thuc.

In [ ]:
SOURCE_DATA_YAML = CLEAN_YAML
TRAINING_ROOT = DATA_ROOT / "processed" / "vietnamese-traffic-signs-kaggle-training-balanced"
TRAINING_YAML = DATA_ROOT / "data.training.yaml"
RUNTIME_DATA_YAML = CACHE_ROOT / "data.training.runtime.yaml"

MIN_TRAIN_OBJECTS_PER_CLASS = 40
MAX_TRAIN_OBJECTS_PER_CLASS = 180
MIN_TEST_OBJECTS_FOR_REPORT = 5

source_cfg = read_yaml(SOURCE_DATA_YAML)
names = normalize_names(source_cfg["names"])

reset_yolo_dirs(TRAINING_ROOT)

source_train_dir = resolve_split_dir(SOURCE_DATA_YAML, "train")
samples = []
original_train_counts = Counter()
for image_path in sorted(path for path in source_train_dir.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS):
    label_path = label_path_for_image(image_path, source_train_dir)
    class_counts = Counter()
    for label in read_yolo_label(label_path):
        if "error" not in label:
            class_counts[int(label["class_id"])] += 1
    if class_counts:
        samples.append({"image_path": image_path, "label_path": label_path, "class_counts": class_counts})
        original_train_counts.update(class_counts)

if not samples:
    raise ValueError("No train samples found after cleaning/remap.")

rng = random.Random(42)
rng.shuffle(samples)
samples.sort(key=lambda sample: min(original_train_counts[cid] for cid in sample["class_counts"]))

selected = []
selected_counts = Counter()
for sample in samples:
    would_exceed = any(
        selected_counts[cid] + count > MAX_TRAIN_OBJECTS_PER_CLASS
        for cid, count in sample["class_counts"].items()
    )
    if not would_exceed:
        selected.append((sample, ""))
        selected_counts.update(sample["class_counts"])

# Guarantee at least one image for every class that exists in the source train split.
selected_paths = {sample["image_path"] for sample, _ in selected}
for cid in names:
    if original_train_counts[cid] > 0 and selected_counts[cid] == 0:
        candidates = [sample for sample in samples if cid in sample["class_counts"]]
        chosen = min(candidates, key=lambda sample: sum(selected_counts[k] for k in sample["class_counts"]))
        selected.append((chosen, ""))
        selected_paths.add(chosen["image_path"])
        selected_counts.update(chosen["class_counts"])

# Oversample rare classes. This duplicates real training images only when the dataset is genuinely too small.
duplicate_index = 0
for cid in names:
    candidates = [sample for sample in samples if cid in sample["class_counts"]]
    if not candidates:
        continue
    candidate_index = 0
    candidates = sorted(candidates, key=lambda sample: sum(original_train_counts[k] for k in sample["class_counts"]))
    while selected_counts[cid] < MIN_TRAIN_OBJECTS_PER_CLASS:
        sample = candidates[candidate_index % len(candidates)]
        duplicate_index += 1
        selected.append((sample, f"__dup{duplicate_index:05d}"))
        selected_counts.update(sample["class_counts"])
        candidate_index += 1

# Prune samples that mostly keep over-represented classes high, while preserving the minimum target for every class.
changed = True
while changed:
    changed = False
    removable_order = sorted(
        range(len(selected)),
        key=lambda idx: (0 if selected[idx][1] else 1, -sum(selected_counts[c] for c in selected[idx][0]["class_counts"])),
    )
    for idx in removable_order:
        sample, suffix = selected[idx]
        class_counts = sample["class_counts"]
        helps_over_cap = any(selected_counts[cid] > MAX_TRAIN_OBJECTS_PER_CLASS for cid in class_counts)
        keeps_minimums = all(selected_counts[cid] - count >= MIN_TRAIN_OBJECTS_PER_CLASS for cid, count in class_counts.items())
        if helps_over_cap and keeps_minimums:
            selected_counts.subtract(class_counts)
            del selected[idx]
            changed = True
            break

for output_index, (sample, suffix) in enumerate(selected):
    image_path = sample["image_path"]
    label_path = sample["label_path"]
    if not suffix:
        out_name = image_path.name
        out_label_name = label_path.name
    else:
        out_name = f"{image_path.stem}{suffix}{image_path.suffix}"
        out_label_name = f"{image_path.stem}{suffix}.txt"
    link_or_copy(image_path, TRAINING_ROOT / "images" / "train" / out_name)
    link_or_copy(label_path, TRAINING_ROOT / "labels" / "train" / out_label_name, use_copy=True)

for split in ["val", "test"]:
    source_dir = resolve_split_dir(SOURCE_DATA_YAML, split)
    for image_path in sorted(path for path in source_dir.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS):
        label_path = label_path_for_image(image_path, source_dir)
        link_or_copy(image_path, TRAINING_ROOT / "images" / split / image_path.name)
        link_or_copy(label_path, TRAINING_ROOT / "labels" / split / label_path.name, use_copy=True)

training_cfg = {
    "path": str(TRAINING_ROOT).replace("\\", "/"),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": names,
}
write_yaml(TRAINING_YAML, training_cfg)

runtime_cfg = read_yaml(TRAINING_YAML)
runtime_root = Path(runtime_cfg.get("path", TRAINING_YAML.parent))
if not runtime_root.is_absolute():
    runtime_root = (TRAINING_YAML.parent / runtime_root).resolve()
runtime_cfg["path"] = str(runtime_root).replace("\\", "/")
write_yaml(RUNTIME_DATA_YAML, runtime_cfg)

images_df, labels_df = scan_yolo_dataset(TRAINING_YAML)
if images_df.empty:
    raise ValueError("No images found after preparing training dataset.")

summary_df = images_df.groupby("split").agg(images=("image_path", "count"), objects=("objects", "sum")).reset_index()
display_styled_table(summary_df, "Prepared YOLO Dataset")

balanced_train_counts = labels_df[labels_df["split"] == "train"].groupby("class_id").size().to_dict()
balance_rows = []
for class_id, class_name in names.items():
    balance_rows.append({
        "class_id": int(class_id),
        "class_name": class_name,
        "original_train_objects": int(original_train_counts.get(class_id, 0)),
        "balanced_train_objects": int(balanced_train_counts.get(class_id, 0)),
        "below_min_target": int(balanced_train_counts.get(class_id, 0)) < MIN_TRAIN_OBJECTS_PER_CLASS,
        "above_max_target": int(balanced_train_counts.get(class_id, 0)) > MAX_TRAIN_OBJECTS_PER_CLASS,
    })
display_styled_table(pd.DataFrame(balance_rows).sort_values("balanced_train_objects"), "Train Object Balancing Report", 120)
print("Runtime data:", RUNTIME_DATA_YAML)

full_class_rows = []
for split in ["train", "val", "test"]:
    split_labels = labels_df[labels_df["split"] == split]
    class_counts = split_labels.groupby("class_id").size().to_dict() if not split_labels.empty else {}
    for class_id, class_name in names.items():
        full_class_rows.append({
            "split": split,
            "class_id": int(class_id),
            "class_name": class_name,
            "objects": int(class_counts.get(int(class_id), 0)),
        })

full_class_df = pd.DataFrame(full_class_rows)
display_styled_table(full_class_df.sort_values(["split", "objects"], ascending=[True, True]), "Object Count by Class", 120)

test_support_df = full_class_df[full_class_df["split"] == "test"].copy()
test_support_df["enough_for_evaluation"] = test_support_df["objects"] >= MIN_TEST_OBJECTS_FOR_REPORT
display_styled_table(
    test_support_df.sort_values(["enough_for_evaluation", "objects"], ascending=[True, True]),
    "Evaluation Support by Class",
    120,
)

## **`6. MODEL TRAINING & VALIDATION`**

In [ ]:
BASE_MODEL = "yolo11n-cls.pt"
model = YOLO(BASE_MODEL)

train_results = model.train(
    data=str(RUNTIME_DATA_ROOT),
    epochs=10,
    imgsz=224,
    batch=32,
    workers=2,
    project=str(PROJECT_ROOT / "runs" / "train"),
    name="traffic-sign-yolo-cls",
    seed=42,
    patience=5,
    exist_ok=True,
    plots=True,
)

## **`7. MODEL EVALUATION`**

In [ ]:
if not BEST_WEIGHT.exists():
    raise FileNotFoundError(f"Missing weight: {BEST_WEIGHT}")

eval_model = YOLO(str(BEST_WEIGHT))

metrics = eval_model.val(
    data=str(RUNTIME_DATA_ROOT),
    split="test",
    imgsz=224,
    project=str(PROJECT_ROOT / "runs" / "eval"),
    name=f"{RUN_NAME}-test",
    plots=True,
)

evaluation_df = pd.DataFrame([{
    "run_name": RUN_NAME,
    "top1": float(metrics.top1),
    "top5": float(metrics.top5),
    "best_weight": str(BEST_WEIGHT),
}])
display_styled_table(evaluation_df, "Evaluation Metrics")

## **`8. INFERENCE`**

In [ ]:
sample = images_df[images_df["split"] == "test"].sample(n=1, random_state=15).iloc[0]
result = eval_model.predict(source=str(sample["image_path"]), imgsz=224, verbose=False)[0]

probs = result.probs
rows = []
for class_id, conf in zip(probs.top5, probs.top5conf):
    class_id = int(class_id)
    rows.append({
        "class_id": class_id,
        "class_name": result.names[class_id],
        "confidence": float(conf),
    })

display_styled_table(pd.DataFrame(rows), "Top Predictions")

In [ ]:
# =========================
# Giao diện nhận diện biển báo giao thông YOLO - Colab
# Có Auto Tune Conf + Img size
# =========================

from ultralytics import YOLO
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image as PILImage
import ipywidgets as widgets
import pandas as pd
import numpy as np

# =========================
# 1. Load model
# =========================
model_path = "/content/runs/train/vietnam-traffic-sign-yolo-balanced/weights/best.pt"
model = YOLO(model_path)

# =========================
# 2. Config
# =========================
AUTO_FOCUS = True
FOCUS_PADDING = 0.25
MAX_DISPLAY_SIZE = (550, 550)

# Nếu bật, hệ thống tự chọn Conf + Img size trước khi predict chính
AUTO_TUNE_DEFAULT = True

upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=True
)

auto_tune_checkbox = widgets.Checkbox(
    value=AUTO_TUNE_DEFAULT,
    description="Tự chọn Conf + Img size"
)

conf_slider = widgets.FloatSlider(
    value=0.25,
    min=0.05,
    max=1.0,
    step=0.05,
    description="Conf:"
)

imgsz_slider = widgets.IntSlider(
    value=416,
    min=320,
    max=1024,
    step=32,
    description="Img size:"
)

predict_button = widgets.Button(
    description="Dự đoán ảnh",
    button_style="success"
)

output = widgets.Output()

# =========================
# 3. Helper functions
# =========================
def save_uploaded_images(upload_value, upload_dir: Path):
    upload_dir.mkdir(exist_ok=True)

    for old_file in upload_dir.glob("*"):
        old_file.unlink()

    image_paths = []

    if isinstance(upload_value, dict):
        for filename, file_info in upload_value.items():
            content = file_info["content"]
            save_path = upload_dir / filename
            with open(save_path, "wb") as f:
                f.write(content)
            image_paths.append(save_path)
    else:
        for file_info in upload_value:
            filename = file_info["name"]
            content = file_info["content"]
            save_path = upload_dir / filename
            with open(save_path, "wb") as f:
                f.write(content)
            image_paths.append(save_path)

    return image_paths


def resize_for_display(pil_img, max_size=MAX_DISPLAY_SIZE):
    img = pil_img.copy()
    img.thumbnail(max_size)
    return img


def crop_focus_area(pil_img, boxes, padding=0.25):
    if boxes is None or len(boxes) == 0:
        return pil_img.copy()

    w, h = pil_img.size

    xyxy_list = []
    for box in boxes:
        xyxy = box.xyxy[0].tolist()
        xyxy_list.append(xyxy)

    xyxy_arr = np.array(xyxy_list)

    x1 = xyxy_arr[:, 0].min()
    y1 = xyxy_arr[:, 1].min()
    x2 = xyxy_arr[:, 2].max()
    y2 = xyxy_arr[:, 3].max()

    box_w = x2 - x1
    box_h = y2 - y1

    pad_x = box_w * padding
    pad_y = box_h * padding

    x1 = max(0, int(x1 - pad_x))
    y1 = max(0, int(y1 - pad_y))
    x2 = min(w, int(x2 + pad_x))
    y2 = min(h, int(y2 + pad_y))

    return pil_img.crop((x1, y1, x2, y2))


def prediction_rows(result, image_name):
    rows = []

    if result.boxes is None or len(result.boxes) == 0:
        return rows

    for i, box in enumerate(result.boxes):
        class_id = int(box.cls.item())
        confidence = float(box.conf.item())
        xyxy = box.xyxy[0].tolist()

        rows.append({
            "Ảnh": image_name,
            "STT": i + 1,
            "Biển báo": result.names[class_id],
            "Độ tin cậy": round(confidence, 3),
            "x1": round(xyxy[0], 1),
            "y1": round(xyxy[1], 1),
            "x2": round(xyxy[2], 1),
            "y2": round(xyxy[3], 1),
        })

    return rows


def show_detection_table(rows, title="Danh sách biển báo phát hiện được"):
    print(title)

    if not rows:
        df = pd.DataFrame([{
            "Kết quả": "Không phát hiện biển báo nào",
            "Gợi ý": "Thử giảm Conf xuống 0.15 - 0.20 hoặc dùng ảnh rõ hơn"
        }])
        display(df)
        return

    df = pd.DataFrame(rows)
    display(df)


def auto_tune_params(image_paths):
    """
    Tự chọn Conf + Img size theo heuristic.

    Ý tưởng:
    - Chạy thử nhiều cặp conf/imgsz.
    - Ưu tiên cấu hình có phát hiện.
    - Ưu tiên confidence trung bình cao.
    - Phạt nhẹ nếu phát hiện quá nhiều box vì có thể là false positive.
    """
    conf_candidates = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
    imgsz_candidates = [416, 512, 640]

    best_score = -999
    best_conf = conf_slider.value
    best_imgsz = imgsz_slider.value
    best_summary = None

    tune_records = []

    for imgsz in imgsz_candidates:
        for conf in conf_candidates:
            results = model.predict(
                source=[str(p) for p in image_paths],
                conf=conf,
                imgsz=imgsz,
                device=0,
                save=False,
                verbose=False
            )

            total_boxes = 0
            total_conf = 0.0
            images_with_detection = 0

            for result in results:
                if result.boxes is not None and len(result.boxes) > 0:
                    confs = result.boxes.conf.tolist()
                    total_boxes += len(confs)
                    total_conf += sum(confs)
                    images_with_detection += 1

            avg_conf = total_conf / total_boxes if total_boxes > 0 else 0.0

            # Heuristic score:
            # - Có ảnh detect được thì cộng mạnh
            # - Confidence cao thì cộng
            # - Box quá nhiều thì phạt nhẹ để tránh nhận lung tung
            # - Img size cao hơn một chút thì cộng nhẹ vì bắt biển nhỏ tốt hơn
            score = (
                images_with_detection * 2.0
                + avg_conf * 2.0
                + min(total_boxes, len(image_paths) * 3) * 0.25
                + (imgsz / 1000) * 0.1
                - max(0, total_boxes - len(image_paths) * 5) * 0.2
            )

            record = {
                "Conf": conf,
                "Img size": imgsz,
                "Số box": total_boxes,
                "Ảnh có detection": images_with_detection,
                "Conf trung bình": round(avg_conf, 3),
                "Score": round(score, 3),
            }
            tune_records.append(record)

            if score > best_score:
                best_score = score
                best_conf = conf
                best_imgsz = imgsz
                best_summary = record

    tune_df = pd.DataFrame(tune_records).sort_values("Score", ascending=False)

    return best_conf, best_imgsz, best_summary, tune_df


def display_original_and_prediction(original_path: Path, result):
    original_img = PILImage.open(original_path).convert("RGB")

    annotated_rgb = result.plot()[:, :, ::-1]
    annotated_img = PILImage.fromarray(annotated_rgb)

    original_show = resize_for_display(original_img)
    annotated_show = resize_for_display(annotated_img)

    original_out = widgets.Output()
    predicted_out = widgets.Output()

    with original_out:
        print("Ảnh ban đầu")
        display(original_show)

    with predicted_out:
        print("Ảnh sau dự đoán")
        display(annotated_show)

    display(widgets.HBox([original_out, predicted_out]))

    if AUTO_FOCUS and result.boxes is not None and len(result.boxes) > 0:
        focus_original = crop_focus_area(original_img, result.boxes, FOCUS_PADDING)
        focus_predicted = crop_focus_area(annotated_img, result.boxes, FOCUS_PADDING)

        focus_original = resize_for_display(focus_original)
        focus_predicted = resize_for_display(focus_predicted)

        focus_original_out = widgets.Output()
        focus_predicted_out = widgets.Output()

        with focus_original_out:
            print("Ảnh ban đầu - vùng focus")
            display(focus_original)

        with focus_predicted_out:
            print("Ảnh dự đoán - vùng focus")
            display(focus_predicted)

        display(widgets.HBox([focus_original_out, focus_predicted_out]))

    rows = prediction_rows(result, original_path.name)
    show_detection_table(rows, title=f"List biển báo trong ảnh: {original_path.name}")

    return rows


# =========================
# 4. Predict function
# =========================
def predict_images(b):
    with output:
        clear_output()

        if len(upload_widget.value) == 0:
            print("Bạn chưa upload ảnh.")
            return

        upload_dir = Path("/content/uploaded_images")
        image_paths = save_uploaded_images(upload_widget.value, upload_dir)

        print(f"Đã upload {len(image_paths)} ảnh.")

        if auto_tune_checkbox.value:
            print("Đang tự chọn Conf + Img size...\n")

            best_conf, best_imgsz, best_summary, tune_df = auto_tune_params(image_paths)

            conf_slider.value = best_conf
            imgsz_slider.value = best_imgsz

            print("Tham số được chọn tự động:")
            print(f"Conf: {best_conf}")
            print(f"Img size: {best_imgsz}")
            print(f"Số box: {best_summary['Số box']}")
            print(f"Conf trung bình: {best_summary['Conf trung bình']}")
            print(f"Score: {best_summary['Score']}")
            print()

            print("Top 5 cấu hình tốt nhất:")
            display(tune_df.head(5))
            print()

        print("Đang dự đoán với tham số cuối...\n")

        results = model.predict(
            source=[str(p) for p in image_paths],
            conf=conf_slider.value,
            imgsz=imgsz_slider.value,
            device=0,
            save=False,
            verbose=False
        )

        print("Kết quả dự đoán:\n")

        all_rows = []

        for image_path, result in zip(image_paths, results):
            print("=" * 100)
            print(f"Ảnh: {image_path.name}")

            rows = display_original_and_prediction(image_path, result)
            all_rows.extend(rows)

        print("=" * 100)
        show_detection_table(all_rows, title="Bảng tổng hợp tất cả biển báo phát hiện được")


predict_button.on_click(predict_images)

# =========================
# 5. Display UI
# =========================
display(widgets.VBox([
    widgets.HTML("<h3>Giao diện nhận diện biển báo giao thông bằng YOLO</h3>"),
    widgets.HTML("<b>Bước 1:</b> Chọn ảnh cần dự đoán"),
    upload_widget,
    widgets.HTML("<b>Bước 2:</b> Chọn chế độ tham số"),
    auto_tune_checkbox,
    widgets.HTML("<b>Nếu tắt auto, chỉnh tay tại đây:</b>"),
    conf_slider,
    imgsz_slider,
    widgets.HTML("<b>Bước 3:</b> Bấm dự đoán"),
    predict_button,
    output
]))